# 01 - Comprehensive Data Audit

Exhaustive data quality checks and exploratory analysis across all pipeline layers:
1. **Raw Data** - JSON files from StatsBomb
2. **Processed Data** - Enriched events, phases, targets
3. **Possessions Data** - Sequence-level features with 360 data
4. **Player Features** - Player defensive actions dataset

This notebook validates the entire data pipeline and identifies any issues before modeling.

In [ ]:
import json
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

REPO_ROOT = Path().resolve().parent
DATA_RAW = REPO_ROOT / 'data' / 'raw'
DATA_PROCESSED = REPO_ROOT / 'data' / 'processed'
DATA_FEATURES = REPO_ROOT / 'data' / 'features'

print(f'Repository: {REPO_ROOT}')
print(f'Raw data exists: {DATA_RAW.exists()}')
print(f'Processed data exists: {DATA_PROCESSED.exists()}')
print(f'Features data exists: {DATA_FEATURES.exists()}')

---
## 1. RAW DATA AUDIT
### 1.1 Competitions Coverage

In [ ]:
# Load and analyze competitions
with open(DATA_RAW / 'competitions.json', 'r', encoding='utf-8') as f:
    competitions = json.load(f)

df_comps = pd.DataFrame(competitions)
print('=== COMPETITIONS ===')
print(f'Total competitions: {len(df_comps)}')
print(f'\nCompetitions by name:')
display(df_comps[['competition_id', 'competition_name', 'season_name', 'competition_gender']].drop_duplicates('competition_id').sort_values('competition_name'))

print(f'\nStatistics:')
print(f'  Unique competitions: {df_comps["competition_name"].nunique()}')
print(f'  Unique seasons: {df_comps["season_name"].nunique()}')
print(f'  Gender coverage: {df_comps["competition_gender"].value_counts().to_dict()}')

### 1.2 Match Coverage

In [ ]:
# Load all matches
match_files = list((DATA_RAW / 'matches').glob('*.json'))
print(f'Total match files: {len(match_files)}')

all_matches = []
for match_file in sorted(match_files):
    with open(match_file, 'r', encoding='utf-8') as f:
        matches = json.load(f)
        all_matches.extend(matches)

df_matches = pd.DataFrame(all_matches)
print(f'Total matches: {len(df_matches)}')

print('\n=== Matches by Competition ===')
matches_by_comp = df_matches.groupby('competition')['match_id'].nunique().sort_values(ascending=False)
display(matches_by_comp)

print(f'\n=== Match Date Range ===')
df_matches['match_date'] = pd.to_datetime(df_matches['match_date'])
print(f'From: {df_matches["match_date"].min()}')
print(f'To: {df_matches["match_date"].max()}')
print(f'Duration: {(df_matches["match_date"].max() - df_matches["match_date"].min()).days} days')

---
## 2. DATA QUALITY SUMMARY

In [ ]:
print('=' * 80)
print('DATA AUDIT COMPLETE')
print('=' * 80)
print(f'✓ Audit complete - Ready for analysis')